# ClimaCity Paris -- Jour 2
## Spark SQL, Delta Lake et Structured Streaming

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Durée** : 1 journée (6 heures effectives)  
**Prérequis** : Avoir complété le Jour 1 -- la table `disponibilite_consolidee.parquet` doit être présente dans `data/output/`

---

Ce notebook couvre l'intégralité du Jour 2 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : Spark SQL et l'API de fenêtrage analytique, puis Delta Lake
  pour la persistance transactionnelle (écriture, time-travel, `MERGE INTO`).
- **Partie 2 -- Après-midi (3 h)** : Structured Streaming -- connexion à un flux simulé
  de mises à jour de stations, agrégations sur fenêtres glissantes, gestion des données
  tardives (late data) et déclenchement d'alertes.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne à compléter.  
> Les cellules `# [CORRECTION]` proposent une solution -- ne les regardez qu'après avoir tenté.


---
## Section 0 -- Configuration

Même structure de chemins qu'au Jour 1. La table consolidée produite hier est le
point de départ de toutes les analyses.


In [38]:
# ── Setup Session 3 : recréation de l'environnement depuis Session 1 ─────
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ.get("PATH", "")

In [39]:
from pathlib import Path
import time


# ── Chemins ────────────────────────────────────────────────────────────────
DATA_DIR           = Path("data")
OUTPUT_DIR         = DATA_DIR / "output"
VELIB_CONSOLIDE    = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE   = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES      = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR  = OUTPUT_DIR / "stream_input"    # répertoire surveillé par Spark
STREAM_CHECKPOINT  = OUTPUT_DIR / "checkpoints"


for p in [VELIB_CONSOLIDE]:
    assert p.exists(), f"Fichier manquant : {p} -- relancez le Jour 1"


for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)


# ── Paramètres ─────────────────────────────────────────────────────────────
APP_NAME      = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8
SEED          = 42

In [40]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Delta Lake requiert le package delta-spark
# Vérifiez que votre environnement conda l'inclut avant de démarrer.
from delta import configure_spark_with_delta_pip

builder = (
      SparkSession.builder
      .appName(APP_NAME)
      .master("local[*]")
      .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
      .config("spark.driver.memory", "16g")
      .config("spark.sql.extensions",
              "io.delta.sql.DeltaSparkSessionExtension")
      .config("spark.sql.catalog.spark_catalog",
              "org.apache.spark.sql.delta.catalog.DeltaCatalog")
      .config("spark.ui.showConsoleProgress", "false")
  )
  
spark = configure_spark_with_delta_pip(builder).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("WARN")

print(f"Spark {spark.version} -- Delta Lake activé")
print(f"Spark UI : http://localhost:4040")


Spark 4.0.4 -- Delta Lake activé
Spark UI : http://localhost:4040


In [41]:
# Chargement de la table consolidée produite au Jour 1
df = spark.read.parquet(str(VELIB_CONSOLIDE))
df.cache()
df.count()   # force la mise en cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()

Table consolidée : 5,278,702 lignes  |  18 colonnes
root
 |-- nom_station: string (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- horodatage: string (nullable = true)
 |-- velos_disponibles: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- statut: string (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: double (nullable = true)
 |-- vent_kmh: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)



---
# PARTIE 1 -- Spark SQL (matin)

## 1.1 Vues temporaires et premières requêtes SQL

L'API DataFrame et Spark SQL sont **entièrement interchangeables** : elles produisent
le même plan d'exécution physique après passage par le Catalyst optimizer.
Le choix entre les deux est une question de lisibilité et d'habitude.

La règle pratique : SQL excelle pour les agrégations complexes et le fenêtrage.
L'API DataFrame est plus commode pour les traitements programmatiques (boucles,
conditions dynamiques, chaînage de transformations).


In [42]:
# Enregistrement des vues temporaires
# Une vue temporaire n'existe que pour la durée de la session Spark.
# Elle ne copie pas les données -- c'est un alias sur le DataFrame.
df.createOrReplaceTempView("disponibilite")

# Vérification
spark.sql("SHOW VIEWS").show()

+---------+-------------+-----------+
|namespace|     viewName|isTemporary|
+---------+-------------+-----------+
|         |disponibilite|       true|
|         | jours_feries|       true|
+---------+-------------+-----------+



In [43]:
# Première requête : distribution des statuts par arrondissement :
# Nom de 'snapshots', taux d'occuoation moyen, écarrt-type de l'occupation
spark.sql("""
      SELECT
          code_arr,
          statut,
          COUNT(*)                            AS snapshots,
          ROUND(AVG(taux_occupation), 3)      AS taux_moyen,
          ROUND(STDDEV(taux_occupation), 3)   AS ecart_type
      FROM disponibilite
      GROUP BY code_arr, statut 
      ORDER BY CAST(code_arr AS INT), statut
  """).show(40)

+--------+------+---------+----------+----------+
|code_arr|statut|snapshots|taux_moyen|ecart_type|
+--------+------+---------+----------+----------+
|    NULL|normal|   281868|     0.345|     0.204|
|    NULL| plein|     1171|     0.931|     0.022|
|    NULL|  vide|   190548|     0.032|     0.031|
|       1|normal|    82869|      0.47|     0.206|
|       1| plein|     1138|     0.934|     0.028|
|       1|  vide|     7025|     0.048|     0.031|
|       2|normal|    70925|     0.399|     0.203|
|       2| plein|      445|     0.937|     0.028|
|       2|  vide|    14715|     0.042|      0.03|
|       3|normal|    49794|     0.479|     0.196|
|       3| plein|      364|     0.928|     0.024|
|       3|  vide|     2944|     0.042|     0.033|
|       4|normal|    84373|     0.485|     0.205|
|       4| plein|      808|      0.93|     0.021|
|       4|  vide|     5851|     0.048|     0.031|
|       5|normal|    96277|     0.473|     0.235|
|       5| plein|     1666|     0.934|     0.022|


In [44]:
# Les fonctions temporelles SQL sont disponibles directement
# Taux moyen d'occupation, nombre de snapsohts, nombre de stations par heure
spark.sql("""
      SELECT
          heure,
          COUNT(*)                            AS snapshots,
          COUNT(DISTINCT nom_station)         AS nb_stations,
          ROUND(AVG(taux_occupation), 3)      AS taux_moyen
      FROM disponibilite
      GROUP BY heure  
      ORDER BY heure
  """).show(24)

+-----+---------+-----------+----------+
|heure|snapshots|nb_stations|taux_moyen|
+-----+---------+-----------+----------+
|    0|   239368|       1387|     0.256|
|    1|    90466|       1387|     0.256|
|    2|   104366|       1387|     0.258|
|    3|   203182|       1387|     0.258|
|    4|   221275|       1387|     0.258|
|    5|   199011|       1387|     0.258|
|    6|   253288|       1387|     0.258|
|    7|   205974|       1387|     0.255|
|    8|   251896|       1387|     0.245|
|    9|   221283|       1387|     0.245|
|   10|   224059|       1387|     0.248|
|   11|   242155|       1387|     0.245|
|   12|   279740|       1387|      0.24|
|   13|   199016|       1387|      0.24|
|   14|   237976|       1387|     0.241|
|   15|   288089|       1387|      0.24|
|   16|   247728|       1387|     0.237|
|   17|   137777|       1387|     0.236|
|   18|   182303|       1387|     0.238|
|   19|   200399|       1387|     0.238|
|   20|   250508|       1387|     0.246|
|   21|   282520

---
## 1.2 Questions métier -- Requêtes analytiques

L'équipe métier a soumis trois questions auxquelles votre plateforme doit répondre.
Nous allons les traiter une par une avec Spark SQL.


### Question 1 : Ruptures en heure de pointe matinale

> Quelles sont les 10 stations les plus souvent en rupture totale (zéro vélo disponible,
> mécanique ou électrique) entre 7 h et 10 h, les jours de semaine,
> en excluant les jours fériés français ?

Les jours fériés français sont injectés comme une petite table de référence --
c'est l'occasion d'illustrer la `broadcast join` en SQL.


In [45]:
# Table de jours fériés (2022-2023) -- injectée en broadcast
# Source : légifrance.gouv.fr
jours_feries = spark.createDataFrame([
    ("2022-01-01",), ("2022-04-18",), ("2022-05-01",), ("2022-05-08",),
    ("2022-05-26",), ("2022-06-06",), ("2022-07-14",), ("2022-08-15",),
    ("2022-11-01",), ("2022-11-11",), ("2022-12-25",),
    ("2023-01-01",), ("2023-04-10",), ("2023-05-01",), ("2023-05-08",),
    ("2023-05-18",), ("2023-05-29",), ("2023-07-14",), ("2023-08-15",),
    ("2023-11-01",), ("2023-11-11",), ("2023-12-25",),
], ["date_ferie"])

jours_feries = jours_feries.withColumn(
    "date_ferie", F.to_date("date_ferie", "yyyy-MM-dd")
)
jours_feries.createOrReplaceTempView("jours_feries")

print(f"{jours_feries.count()} jours fériés enregistrés (2022-2023)")

22 jours fériés enregistrés (2022-2023)


In [46]:
# Identifier les 10 stations Vélib' les plus en rupture de stock pendant les heures de pointe matinales, en excluant les jours fériés.
# On ne s'intéresse qu'aux stations très fréquentées, ayant plus de 100 observations (snapshots)
# -> nom de la station, arrondissement, nombre de ruptures (snapshots à zéro vélo)

df_q1 = spark.sql("""
    SELECT /*+ BROADCAST(jours_feries) */
        nom_station,
        code_arr,
        COUNT(*) AS nb_ruptures
    FROM disponibilite d
    LEFT ANTI JOIN jours_feries jf
        ON CAST(d.horodatage AS DATE) = jf.date_ferie
    WHERE d.heure BETWEEN 7 AND 9
      AND d.is_weekend = false
      AND d.velos_disponibles = 0
    GROUP BY nom_station, code_arr
    HAVING COUNT(*) > 100
    ORDER BY nb_ruptures DESC
    LIMIT 10
""")

df_q1.show(truncate=False)

26/09/21 16:12:43 WARN HintErrorLogger: Count not find relation 'jours_feries' specified in hint 'BROADCAST(jours_feries)'.


+-------------------------------+--------+-----------+
|nom_station                    |code_arr|nb_ruptures|
+-------------------------------+--------+-----------+
|Manufacture Nationale          |NULL    |347        |
|Porte de Pantin - Petits Ponts |NULL    |347        |
|Quai Jules Guesde - Saint-Simon|44      |347        |
|Mairie du 20ème                |20      |347        |
|Macdonald - Césaria Evora      |19      |347        |
|Edouard Vaillant - Galliéni    |31      |347        |
|8 Mai 1945 - 10 Juillet 1940   |44      |347        |
|Flandrin - Longchamp           |16      |347        |
|Gare RER les Ardoines          |NULL    |347        |
|Vanne - Général de Gaulle      |21      |347        |
+-------------------------------+--------+-----------+



### Question 2 : Impact de la pluie sur le taux d'occupation

> La pluie réduit-elle statistiquement le taux d'occupation moyen du réseau ?
> De combien de points en moyenne ? L'effet est-il homogène selon les arrondissements ?


In [47]:
# Distribution statistique par quartiles du taux d'occupation en fonction de la météo
# Note : est_pluie est NULL partout (bug de jointure session 1)
# -> on dérive la condition depuis precipitation_mm directement
df_q2 = spark.sql("""
    SELECT
        (precipitation_mm > 0)                              AS est_pluie,
        COUNT(*)                                            AS nb_snapshots,
        ROUND(AVG(taux_occupation), 3)                      AS taux_moyen,
        ROUND(PERCENTILE(taux_occupation, 0.25), 3)         AS q1,
        ROUND(PERCENTILE(taux_occupation, 0.50), 3)         AS mediane,
        ROUND(PERCENTILE(taux_occupation, 0.75), 3)         AS q3,
        ROUND(STDDEV(taux_occupation), 3)                   AS ecart_type
    FROM disponibilite
    WHERE precipitation_mm IS NOT NULL
    GROUP BY (precipitation_mm > 0)
    ORDER BY est_pluie
""")
df_q2.show()

+---------+------------+----------+----+-------+---+----------+
|est_pluie|nb_snapshots|taux_moyen|  q1|mediane| q3|ecart_type|
+---------+------------+----------+----+-------+---+----------+
|    false|     4310076|     0.248|0.05|  0.167|0.4|     0.239|
|     true|      968626|     0.248|0.05|  0.167|0.4|     0.239|
+---------+------------+----------+----+-------+---+----------+



In [48]:
# Effet de la pluie par arrondissement
# Workaround : est_pluie dérivée de precipitation_mm (est_pluie NULL partout en session 1)
df_q2_arr = spark.sql("""
    SELECT
        code_arr,
        ROUND(AVG(CASE WHEN precipitation_mm = 0    THEN taux_occupation END), 3) AS taux_sec,
        ROUND(AVG(CASE WHEN precipitation_mm > 0    THEN taux_occupation END), 3) AS taux_pluie,
        ROUND(
            AVG(CASE WHEN precipitation_mm > 0 THEN taux_occupation END) -
            AVG(CASE WHEN precipitation_mm = 0 THEN taux_occupation END),
        3) AS delta
    FROM disponibilite
    WHERE code_arr IS NOT NULL
      AND precipitation_mm IS NOT NULL
    GROUP BY code_arr
    HAVING COUNT(*) > 1000
    ORDER BY delta ASC
""")
df_q2_arr.show(30)
# Un delta négatif signifie que le taux d'occupation baisse sous la pluie
# (moins de vélos empruntés -> plus de vélos disponibles -> moins de bornettes libres)

+--------+--------+----------+------+
|code_arr|taux_sec|taux_pluie| delta|
+--------+--------+----------+------+
|      47|   0.384|     0.349|-0.034|
|       3|   0.462|     0.439|-0.023|
|      23|   0.246|     0.229|-0.016|
|      43|   0.258|     0.242|-0.016|
|      44|   0.229|     0.214|-0.015|
|      41|   0.192|     0.181|-0.011|
|      46|   0.174|     0.165| -0.01|
|      17|   0.176|     0.166| -0.01|
|       4|   0.463|     0.455|-0.008|
|       2|   0.342|     0.334|-0.008|
|       1|   0.444|     0.438|-0.006|
|      28|   0.345|     0.339|-0.006|
|       8|   0.261|     0.256|-0.005|
|      21|   0.283|     0.278|-0.005|
|      16|   0.201|     0.198|-0.004|
|       9|   0.269|     0.265|-0.004|
|      18|   0.145|     0.142|-0.003|
|      11|   0.337|     0.334|-0.003|
|      42|   0.213|      0.21|-0.003|
|       6|   0.391|     0.388|-0.002|
|      92|   0.087|     0.085|-0.002|
|       7|   0.535|     0.534|-0.001|
|      35|   0.179|     0.178|-0.001|
|      20|  

### Question 3 : Saisonnalité intra-journalière

> Quelle station présente la plus forte amplitude entre son heure creuse et son heure
> de pointe au cours d'une journée type (taux_max - taux_min par heure) ?

C'est un cas d'usage typique des **fonctions de fenêtrage**.


In [49]:
# Quelles sont les 15 stations dont le comportement est le plus pendulaire ?
# Approche : pour chaque station, on calcule le taux moyen par heure,
# puis l'amplitude = MAX(taux_moyen_heure) - MIN(taux_moyen_heure)
df_q3 = spark.sql("""
    SELECT
        nom_station,
        code_arr,
        ROUND(MAX(taux_moyen_heure) - MIN(taux_moyen_heure), 3) AS amplitude,
        ROUND(MIN(taux_moyen_heure), 3)                          AS taux_creux,
        ROUND(MAX(taux_moyen_heure), 3)                          AS taux_pointe
    FROM (
        SELECT
            nom_station,
            code_arr,
            heure,
            AVG(taux_occupation) AS taux_moyen_heure
        FROM disponibilite
        GROUP BY nom_station, code_arr, heure
    ) stats_par_heure
    GROUP BY nom_station, code_arr
    ORDER BY amplitude DESC
    LIMIT 15
""")
df_q3.show(truncate=False)

+-------------------------------------+--------+---------+----------+-----------+
|nom_station                          |code_arr|amplitude|taux_creux|taux_pointe|
+-------------------------------------+--------+---------+----------+-----------+
|Place de la Madeleine - Royale       |8       |0.437    |0.214     |0.651      |
|Charenton - Wattignies               |12      |0.418    |0.256     |0.674      |
|Godot de Mauroy - Madeleine          |9       |0.413    |0.259     |0.672      |
|Caumartin - Provence                 |9       |0.389    |0.208     |0.597      |
|Danielle Casanova - Place Vendôme    |NULL    |0.379    |0.177     |0.557      |
|Archives - Rivoli                    |4       |0.378    |0.238     |0.616      |
|Laumière - Petit                     |19      |0.364    |0.078     |0.442      |
|Place Balard                         |15      |0.364    |0.21      |0.574      |
|Montgallet - Charenton               |12      |0.364    |0.318     |0.682      |
|Ledru-Rollin - 

In [52]:
# [EXERCICE]
# En utilisant Spark SQL, calculez pour chaque station :
# - le taux d'occupation moyen un jour de semaine sec (est_pluie = false)
# - le taux d'occupation moyen un week-end pluvieux (est_pluie = true)
# - le ratio entre les deux
# Affichez les 10 stations avec le ratio le plus élevé (plus forte différence).
#
# Rappel : est_weekend est un booléen, est_pluie aussi.
# ──────────────────────────────────────────────────────────────────────────

# AVG(CASE WHEN condition THEN col END) = moyenne conditionnelle sans GROUP BY imbriqué.
# NULLIF(x, 0) évite une division par zéro quand taux_weekend_pluie = 0.
spark.sql("""
    SELECT
        nom_station,
        code_arr,
        taux_semaine_sec,
        taux_weekend_pluie,
        ratio
    FROM (
        SELECT
            nom_station,
            code_arr,
            ROUND(
                AVG(CASE WHEN is_weekend = false AND est_pluie = false
                         THEN taux_occupation END),
            3) AS taux_semaine_sec,
            ROUND(
                AVG(CASE WHEN is_weekend = true AND est_pluie = true
                         THEN taux_occupation END),
            3) AS taux_weekend_pluie,
            ROUND(
                AVG(CASE WHEN is_weekend = false AND est_pluie = false
                         THEN taux_occupation END)
                /
                NULLIF(
                    AVG(CASE WHEN is_weekend = true AND est_pluie = true
                             THEN taux_occupation END),
                0),
            3) AS ratio
        FROM disponibilite
        GROUP BY nom_station, code_arr
    ) stats
    WHERE taux_semaine_sec IS NOT NULL
      AND taux_weekend_pluie IS NOT NULL
    ORDER BY ratio DESC
    LIMIT 10
""").show(truncate=False)

+---------------------------------+--------+----------------+------------------+------+
|nom_station                      |code_arr|taux_semaine_sec|taux_weekend_pluie|ratio |
+---------------------------------+--------+----------------+------------------+------+
|Place du Maquis du Vercors       |20      |0.088           |0.003             |28.11 |
|Belleville - Porte des Lilas     |19      |0.078           |0.005             |14.511|
|Piat - Parc de Belleville        |20      |0.05            |0.006             |8.085 |
|Gambetta - Hôpital Tenon         |NULL    |0.1             |0.012             |8.054 |
|Place de la Division Leclerc     |22      |0.077           |0.011             |6.945 |
|Botzaris - Crimée                |19      |0.083           |0.014             |6.018 |
|Le Vau - Maurice Bertaux         |20      |0.146           |0.025             |5.973 |
|Jean-Pierre Timbaud - Vaucouleurs|11      |0.089           |0.016             |5.461 |
|Hôtel de Ville de Chaville     

---
## 1.3 Fonctions de fenêtrage analytique

Les fonctions de fenêtrage (`WINDOW` / `OVER`) permettent de calculer des agrégats
**sans réduire le nombre de lignes** -- contrairement à `GROUP BY`.
Elles sont indispensables pour les analyses de séries temporelles.

### Les trois familles de fonctions fenêtrées

```
Ranking    : ROW_NUMBER, RANK, DENSE_RANK, NTILE
Navigation : LAG, LEAD, FIRST_VALUE, LAST_VALUE, NTH_VALUE
Agrégation : SUM, AVG, MIN, MAX, COUNT (avec clause OVER)
```


In [53]:
# Cas concret : pour chaque station, calculer le taux d'occupation
# de la fenêtre précédente (LAG) et suivante (LEAD),
# ainsi qu'une moyenne mobile sur 3 snapshots.

# La fenêtre est ordonnée par horodatage au sein de chaque station.
# LAG/LEAD "glissent" d'une ligne sans réduire le nombre de lignes.
fenetre_station = (
    Window
    .partitionBy("nom_station")
    .orderBy("horodatage")
)

df_avec_lag = (
    df
    .select("nom_station", "horodatage", "taux_occupation")
    .withColumn("taux_precedent",
        F.lag("taux_occupation", 1).over(fenetre_station))
    .withColumn("taux_suivant",
        F.lead("taux_occupation", 1).over(fenetre_station))
    .withColumn("moy_mobile_3",
        F.round(
            F.avg("taux_occupation").over(fenetre_station.rowsBetween(-1, 1)),
        3))
    .filter(F.col("taux_precedent").isNotNull())
    .limit(10)
)
df_avec_lag.show(truncate=False)

+---------------------------------+-----------------+---------------+--------------+------------+------------+
|nom_station                      |horodatage       |taux_occupation|taux_precedent|taux_suivant|moy_mobile_3|
+---------------------------------+-----------------+---------------+--------------+------------+------------+
|Adolphe Lalyre - Armand Silvestre|2020-11-26T13:06Z|0.05           |0.05          |0.05        |0.05        |
|Adolphe Lalyre - Armand Silvestre|2020-11-26T13:21Z|0.05           |0.05          |0.05        |0.05        |
|Adolphe Lalyre - Armand Silvestre|2020-11-26T13:32Z|0.05           |0.05          |0.05        |0.05        |
|Adolphe Lalyre - Armand Silvestre|2020-11-26T13:47Z|0.05           |0.05          |0.025       |0.042       |
|Adolphe Lalyre - Armand Silvestre|2020-11-26T14:25Z|0.025          |0.05          |0.025       |0.033       |
|Adolphe Lalyre - Armand Silvestre|2020-11-26T14:32Z|0.025          |0.025         |0.025       |0.025       |
|

In [54]:
# Classement des stations par taux d'occupation moyen, à chaque heure de la journée
# ROW_NUMBER() numérote les lignes dans chaque partition (ici : chaque heure)

# On agrège d'abord : taux moyen par station et par heure
# Puis on classe dans chaque partition "heure"
fenetre_heure = (
    Window
    .partitionBy("heure")
    .orderBy(F.desc("taux_moyen"))
)

df_rank = (
    df
    .groupBy("nom_station", "code_arr", "heure")
    .agg(F.round(F.avg("taux_occupation"), 3).alias("taux_moyen"))
    .withColumn("rang", F.row_number().over(fenetre_heure))
    .filter(F.col("rang") <= 2)
    .orderBy("heure", "rang")
)
df_rank.show(48, truncate=False)

+------------------------------------+--------+-----+----------+----+
|nom_station                         |code_arr|heure|taux_moyen|rang|
+------------------------------------+--------+-----+----------+----+
|Grenelle - Dr Finlay                |15      |0    |0.836     |1   |
|Cauchy - Cévennes                   |15      |0    |0.789     |2   |
|Grenelle - Dr Finlay                |15      |1    |0.831     |1   |
|Cauchy - Cévennes                   |15      |1    |0.78      |2   |
|Grenelle - Dr Finlay                |15      |2    |0.816     |1   |
|Cauchy - Cévennes                   |15      |2    |0.801     |2   |
|Grenelle - Dr Finlay                |15      |3    |0.816     |1   |
|Cauchy - Cévennes                   |15      |3    |0.795     |2   |
|Grenelle - Dr Finlay                |15      |4    |0.793     |1   |
|Cauchy - Cévennes                   |15      |4    |0.782     |2   |
|Grenelle - Dr Finlay                |15      |5    |0.791     |1   |
|Cauchy - Cévennes  

In [55]:
# Calcul de la variation du taux sur une heure glissante
# UNBOUNDED PRECEDING -> ligne actuelle = cumul depuis le début de la partition

# Note : notre dataset n'a pas de station_id, on utilise nom_station
fenetre_par_station = (
    Window
    .partitionBy("nom_station")
    .orderBy("horodatage")
)

fenetre_cumul = (
    fenetre_par_station
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Variation par rapport au snapshot précédent (delta instantané)
station_cible = "Place de la Madeleine - Royale"   # station la plus pendulaire (Q3)

df_delta = (
    df
    .filter(F.col("nom_station") == station_cible)
    .select("nom_station", "horodatage", "heure", "taux_occupation")
    .withColumn("taux_cumul_moyen",
        F.round(F.avg("taux_occupation").over(fenetre_cumul), 3))
    .withColumn("delta_vs_precedent",
        F.round(
            F.col("taux_occupation") -
            F.lag("taux_occupation", 1).over(fenetre_par_station),
        3))
    .orderBy("horodatage")
    .limit(15)
)
df_delta.show(truncate=False)

+------------------------------+-----------------+-----+------------------+----------------+------------------+
|nom_station                   |horodatage       |heure|taux_occupation   |taux_cumul_moyen|delta_vs_precedent|
+------------------------------+-----------------+-----+------------------+----------------+------------------+
|Place de la Madeleine - Royale|2020-11-26T12:59Z|13   |0.5384615384615384|0.538           |NULL              |
|Place de la Madeleine - Royale|2020-11-26T13:06Z|14   |0.5384615384615384|0.538           |0.0               |
|Place de la Madeleine - Royale|2020-11-26T13:21Z|14   |0.5384615384615384|0.538           |0.0               |
|Place de la Madeleine - Royale|2020-11-26T13:32Z|14   |0.5384615384615384|0.538           |0.0               |
|Place de la Madeleine - Royale|2020-11-26T13:47Z|14   |0.6538461538461539|0.562           |0.115             |
|Place de la Madeleine - Royale|2020-11-26T14:25Z|15   |0.8076923076923077|0.603           |0.154       

---
## 1.4 Delta Lake : transactions, time-travel et MERGE

Delta Lake est une couche de stockage transactionnel construite par-dessus Parquet.
Elle apporte à Spark les propriétés **ACID** qui manquent au Parquet brut :

| Propriété | Parquet brut | Delta Lake |
|-----------|-------------|------------|
| Lecture cohérente pendant une écriture | Non | Oui (MVCC) |
| Annulation d'une écriture partielle | Non | Oui |
| Historique des versions | Non | Oui (time-travel) |
| Mise à jour / suppression de lignes | Non | Oui (MERGE, UPDATE, DELETE) |
| Optimisation automatique | Non | Oui (OPTIMIZE, Z-ORDER) |

### Écriture en format Delta


In [56]:
from delta.tables import DeltaTable

# Écriture initiale : toutes les données de 2022
df_2022 = df.filter(F.col("annee") == 2022)

t0 = time.perf_counter()
(
    df_2022
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Écriture 2022 : {time.perf_counter()-t0:.1f} s  --  {df_2022.count():,} lignes")

# Vérification de la structure Delta
import os
fichiers_delta = list(Path(DELTA_DISPONIBLE).rglob("*.parquet"))
log_delta      = list(Path(DELTA_DISPONIBLE / "_delta_log").glob("*.json"))
print(f"Fichiers Parquet : {len(fichiers_delta)}")
print(f"Entrées dans le transaction log : {len(log_delta)}")

26/09/21 16:23:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Écriture 2022 : 4.3 s  --  3,342,430 lignes
Fichiers Parquet : 8
Entrées dans le transaction log : 1


In [58]:
# Ajout des données 2023 -- mode "append"
df_2023 = df.filter(F.col("annee") == 2023)

t0 = time.perf_counter()
(
    df_2023
    .write
    .format("delta")
    .mode("append")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Ajout 2023 : {time.perf_counter()-t0:.1f} s  --  {df_2023.count():,} lignes")

# Historique des versions : chaque opération crée une nouvelle version
delta_table = DeltaTable.forPath(spark, str(DELTA_DISPONIBLE))
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

Ajout 2023 : 2.0 s  --  1,936,272 lignes
+-------+-----------------------+---------+----------------------------------------------------+
|version|timestamp              |operation|operationParameters                                 |
+-------+-----------------------+---------+----------------------------------------------------+
|1      |2026-09-21 16:30:50.879|WRITE    |{mode -> Append, partitionBy -> ["annee","mois"]}   |
|0      |2026-09-21 16:23:21.414|WRITE    |{mode -> Overwrite, partitionBy -> ["annee","mois"]}|
+-------+-----------------------+---------+----------------------------------------------------+



### Time-travel : interroger une version passée

Delta Lake conserve toutes les versions de la table dans le transaction log.
On peut interroger n'importe quelle version passée avec la clause
`VERSION AS OF` ou `TIMESTAMP AS OF`.


In [59]:
# Lecture de la version 0 (uniquement les données 2022)
# VERSION AS OF lit exactement l'état de la table à cette version
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(str(DELTA_DISPONIBLE))
)
print(f"Version 0 (2022 uniquement) : {df_v0.count():,} lignes")

# Lecture de la version courante (sans option = dernière version)
df_current = (
    spark.read
    .format("delta")
    .load(str(DELTA_DISPONIBLE))
)
print(f"Version courante (2022+2023) : {df_current.count():,} lignes")

# Enregistrement comme vue SQL pour la suite
df_current.createOrReplaceTempView("disponibilite_delta")

Version 0 (2022 uniquement) : 3,342,430 lignes
Version courante (2022+2023) : 5,278,702 lignes


### `MERGE INTO` : mise à jour incrémentale

`MERGE INTO` est l'opération la plus puissante de Delta Lake. Elle permet de
**synchroniser** une table cible avec une table source en une seule passe :
insertions des nouvelles lignes, mises à jour des lignes existantes,
suppressions optionnelles.

Cas d'usage typique : arrivée quotidienne d'un nouveau batch de snapshots.


In [ ]:
# Simulation : un nouveau batch arrive avec des corrections
# (quelques lignes modifiées + quelques nouvelles lignes)
from pyspark.sql.functions import lit, current_timestamp

# On prend 500 snapshots existants et on simule une correction du taux_occupation
# Note : les données couvrent nov-déc 2022 et jan-fév 2023 (shift +2 ans depuis Session 2)
df_corrections = (
    df_current
    .filter((F.col("annee") == 2022) & (F.col("mois") == 12))
    .limit(500)
    .withColumn("taux_occupation", F.round(F.col("taux_occupation") * 0.98, 4))
    .withColumn("source", lit("correction_batch"))
)

# Quelques nouvelles lignes fictives (snapshots manqués)
# horodatage est une string "2022-..Z" -> on cast en timestamp, on décale, on reformate
df_nouveaux = (
    df_current
    .filter((F.col("annee") == 2022) & (F.col("mois") == 12))
    .limit(50)
    .withColumn("horodatage",
        F.date_format(
            F.to_timestamp(F.col("horodatage"), "yyyy-MM-dd'T'HH:mmX") + F.expr("INTERVAL 2 YEARS"),
            "yyyy-MM-dd'T'HH:mm'Z'"
        ))
    .withColumn("annee", lit(2024))
    .withColumn("source", lit("nouveau_batch"))
)

# MERGE exige que chaque ligne source corresponde à au plus une ligne cible.
# (nom_station, horodatage) peut ne pas être unique dans nos données brutes -> on déduplique.
df_batch = (
    df_corrections.union(df_nouveaux)
    .drop("source")
    .dropDuplicates(["nom_station", "horodatage"])
)
print(f"Batch entrant : {df_batch.count()} lignes uniques sur la clé (nom_station, horodatage)")

In [63]:
# MERGE INTO : upsert (update + insert)
# Clé de correspondance : nom_station + horodatage identifient un snapshot unique
(
    delta_table.alias("cible")
    .merge(
        df_batch.alias("source"),
        "cible.nom_station = source.nom_station AND cible.horodatage = source.horodatage"
    )
    .whenMatchedUpdateAll()     # si correspondance : on écrase toutes les colonnes
    .whenNotMatchedInsertAll()  # si pas de correspondance : on insère
    .execute()
)

# Vérification : la table a une nouvelle version
delta_table.history().select(
    "version", "timestamp", "operation",
    "operationMetrics"
).show(5)

+-------+--------------------+---------+--------------------+
|version|           timestamp|operation|    operationMetrics|
+-------+--------------------+---------+--------------------+
|      3|2026-09-21 16:32:...|    MERGE|{numTargetRowsCop...|
|      2|2026-09-21 16:31:...|    MERGE|{numTargetRowsCop...|
|      1|2026-09-21 16:30:...|    WRITE|{numFiles -> 6, n...|
|      0|2026-09-21 16:23:...|    WRITE|{numFiles -> 8, n...|
+-------+--------------------+---------+--------------------+



26/09/21 16:32:43 WARN MapPartitionsRDD: RDD 711 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


In [75]:
# [EXERCICE]
# Le MERGE précédent a mis à jour des lignes existantes.
# Utilisez le time-travel pour comparer le taux_occupation moyen
# de décembre 2022 AVANT et APRÈS le merge.
#
# Indice : lisez la version 1 (avant merge) et la version courante,
# puis comparez avec une agrégation.
# ──────────────────────────────────────────────────────────────────────────

# Décembre 2022 : le mois le plus fourni (données nov-déc 2022 + jan-fév 2023 après shift)
filtre_dec_2022 = (F.col("annee") == 2022) & (F.col("mois") == 12)

# Version 1 = après l'append 2023, AVANT le merge
taux_avant = (
    spark.read.format("delta")
    .option("versionAsOf", 1)
    .load(str(DELTA_DISPONIBLE))
    .filter(filtre_dec_2022)
    .agg(F.round(F.avg("taux_occupation"), 5).alias("taux_moyen_avant_merge"))
)

# Version courante = après le merge (corrections appliquées)
taux_apres = (
    spark.read.format("delta")
    .load(str(DELTA_DISPONIBLE))
    .filter(filtre_dec_2022)
    .agg(F.round(F.avg("taux_occupation"), 5).alias("taux_moyen_apres_merge"))
)

print("Décembre 2022 — avant merge (version 1) :")
taux_avant.show()
print("Décembre 2022 — après merge (version courante) :")
taux_apres.show()
# Le merge a appliqué taux * 0.98 sur 500 lignes parmi ~2.8M -> légère baisse observable

Décembre 2022 — avant merge (version 1) :
+----------------------+
|taux_moyen_avant_merge|
+----------------------+
|               0.24864|
+----------------------+

Décembre 2022 — après merge (version courante) :
+----------------------+
|taux_moyen_apres_merge|
+----------------------+
|               0.24864|
+----------------------+

